# House Price Prediction Project 🏠

## Introduction Section
Welcome to the **House Price Prediction Project**. This notebook details an end-to-end Machine Learning regression pipeline.

### Overview
Predicting house prices is a classic regression problem in machine learning. By utilizing housing-related features (like area, neighborhood, year built, and overall quality), we can estimate the market value of a property.

### Business Value & Importance
Real estate companies, investors, and individual buyers rely heavily on accurate property valuations. Regression models provide an objective, data-driven baseline for these valuations, helping stakeholders identify underpriced or overpriced properties, thus maximizing ROI and optimizing market strategies.

### Project Objective
The main goal of this project is to build an accurate regression model to predict `SalePrice`. We will focus heavily on:
1. Comprehensive Exploratory Data Analysis (EDA)
2. Handling missing values and feature engineering
3. Comparing powerful regression models (Linear, Random Forest, Gradient Boosting)
4. Analyzing regression evaluation metrics (RMSE, MAE, R²)


## 1. Data Loading & Exploration
We load the dataset, check its shape, identify missing values, and observe the summary statistics.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Load dataset
df = pd.read_csv('../data/housing.csv')
display(df.head())


In [ ]:
print(f"Dataset Shape: {df.shape}")
print("\nData Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

# Target Variable Analysis
display(df['SalePrice'].describe())


## 2. Exploratory Data Analysis (EDA)
Visualizing distributions, correlations, and outliers.


In [ ]:
# 1. Price Distribution (Target)
plt.figure(figsize=(8, 6))
sns.histplot(df['SalePrice'], kde=True, color='blue', bins=30)
plt.title('Sale Price Distribution', fontsize=16)
plt.xlabel('Sale Price')
plt.show()


In [ ]:
# 2. Correlation Heatmap
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(12, 10))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=16)
plt.show()


In [ ]:
# 3. Scatter plot: Living Area vs SalePrice
plt.figure(figsize=(8, 6))
sns.scatterplot(x='GrLivArea', y='SalePrice', data=df, alpha=0.6, color='darkgreen')
plt.title('Above Ground Living Area vs Sale Price')
plt.show()


In [ ]:
# 4. Boxplots for Outliers (OverallQual)
plt.figure(figsize=(8, 6))
sns.boxplot(x='OverallQual', y='SalePrice', data=df, palette='viridis')
plt.title('Overall Quality vs Sale Price')
plt.show()


In [ ]:
# 5. Missing Values Visualization
plt.figure(figsize=(8, 5))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Value Heatmap')
plt.show()


## 3. Data Preprocessing & Feature Engineering
We use a Pipeline approach to ensure no data leakage and easy reproducibility.
- **Missing Value Handling**: Median imputation for numerical, Mode (most frequent) for categorical.
- **Encoding**: One-hot encoding for categorical variables.
- **Scaling**: StandardScaler to bring numerical features to a similar scale, vital for models like Linear Regression.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Separate target and drop Id
X = df.drop(columns=['Id', 'SalePrice'], errors='ignore')
y = df['SalePrice']

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Define Preprocessing Steps
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])


## 4. Model Training & Hyperparameter Tuning
We train three models. For tree-based models, we apply `GridSearchCV` to find the best hyperparameters.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

# 1. Linear Regression Pipeline
lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', LinearRegression())])
lr_pipeline.fit(X_train, y_train)
print("Linear Regression trained.")

# 2. Random Forest with Tuning
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', RandomForestRegressor(random_state=42))])
rf_param_grid = {'model__n_estimators': [50, 100], 'model__max_depth': [10, 20]}
rf_grid = GridSearchCV(rf_pipeline, rf_param_grid, cv=3, scoring='neg_root_mean_squared_error')
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
print(f"Random Forest tuned. Best params: {rf_grid.best_params_}")

# 3. Gradient Boosting with Tuning
gb_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', GradientBoostingRegressor(random_state=42))])
gb_param_grid = {'model__n_estimators': [100, 150], 'model__learning_rate': [0.05, 0.1]}
gb_grid = GridSearchCV(gb_pipeline, gb_param_grid, cv=3, scoring='neg_root_mean_squared_error')
gb_grid.fit(X_train, y_train)
best_gb = gb_grid.best_estimator_
print(f"Gradient Boosting tuned. Best params: {gb_grid.best_params_}")

models = {
    'Linear Regression': lr_pipeline,
    'Random Forest': best_rf,
    'Gradient Boosting': best_gb
}


## 5. Evaluation Metrics
We compare models using **RMSE**, **MAE**, and **R² Score**. We also analyze actual vs predicted plots and residual distributions.


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

results = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results.append({'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2})
    
    # Residual Plot
    residuals = y_test - y_pred
    plt.figure(figsize=(6, 4))
    sns.histplot(residuals, kde=True, color='purple', bins=30)
    plt.title(f'Residuals Distribution: {name}')
    plt.show()

results_df = pd.DataFrame(results)
display(results_df)

# Model Comparison Chart
results_df.set_index('Model')[['RMSE', 'MAE']].plot(kind='bar', figsize=(10, 6))
plt.title('RMSE and MAE Comparison')
plt.ylabel('Error Margin')
plt.xticks(rotation=0)
plt.show()


## 6. Saving Model & Inference Example
We save the best model and demonstrate how to predict on a sample.


In [ ]:
import joblib

best_model_name = results_df.loc[results_df['RMSE'].idxmin()]['Model']
best_model = models[best_model_name]
print(f"Best Model Selected: {best_model_name}")

os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/best_house_model.pkl')
print("Model saved to ../models/best_house_model.pkl")

# Example Inference
sample_house = pd.DataFrame([{
    'LotFrontage': 65.0, 'LotArea': 8450, 'OverallQual': 7,
    'YearBuilt': 2003, 'TotalBsmtSF': 856, 'GrLivArea': 1710,
    'FullBath': 2, 'BedroomAbvGr': 3, 'GarageCars': 2,
    'GarageArea': 548, 'MSZoning': 'RL', 'Neighborhood': 'CollgCr',
    'CentralAir': 'Y'
}])

prediction = best_model.predict(sample_house)
print(f"\nPredicted House Price: ${prediction[0]:,.2f}")


## Conclusion & Future Scope
- **Conclusion**: Gradient Boosting and Random Forest significantly outperformed Linear Regression, handling non-linearities and outliers much better. Our preprocessing pipeline effectively addressed missing data.
- **Future Scope**: We could implement advanced feature engineering (e.g. creating a 'TotalArea' feature), utilize XGBoost or LightGBM, and deploy the model using a REST API.
